# 04 - Repurchase Prediction

This notebook predicts whether a customer will make a repeat purchase within a fixed prediction window, using only information available up to the end of an observation window — the central discipline being avoiding data leakage from the future into the features.

## Research Questions

1. Who is likely to repurchase within the next 90 days, and at what decision threshold should we act on that prediction?
2. Does the model add value over a naive baseline, once evaluated honestly on time-based (not random) splits?

## Plan

0. Setup & Loading
1. Decision — Observation / Prediction Window Split
2. Feature Engineering (observation window only)
3. Target Construction
4. Train / Test Split (time-based)
5. Baseline Models
6. Model Comparison & Evaluation
7. Threshold Optimization
8. Feature Importance


# 0. Setup & Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)
from sklearn.calibration import calibration_curve
from pathlib import Path
from IPython.display import display, Markdown

PROC = Path("../data/processed")
clean = pd.read_parquet(PROC / "clean_transactions.parquet")
purchases = clean[~clean["IsCancellation"]].copy()

display(Markdown(f"""
### Input
* **Purchase rows:** `{len(purchases):,}`
* **Date range:** `{purchases['InvoiceDate'].min()}` to `{purchases['InvoiceDate'].max()}`
"""))


# 1. Decision — Observation / Prediction Window Split

To predict repurchase without leaking future information, the timeline is split in two:

- **Observation window**: all data up to a cutoff date. Features are computed *only* from this window.
- **Prediction window**: the 90 days immediately after the cutoff. The target is whether the customer purchases at least once in this window.

In [ ]:
PREDICTION_WINDOW_DAYS = 90

max_date = purchases["InvoiceDate"].max()
CUTOFF_DATE = max_date - pd.Timedelta(PREDICTION_WINDOW_DAYS, unit="D")

display(Markdown(f"""
* **Dataset end date:** `{max_date}`
* **Prediction window:** `{PREDICTION_WINDOW_DAYS}` days
* **Cutoff date (end of observation window):** `{CUTOFF_DATE}`
"""))

obs = purchases[purchases["InvoiceDate"] <= CUTOFF_DATE].copy()
pred_window = purchases[
    (purchases["InvoiceDate"] > CUTOFF_DATE) & (purchases["InvoiceDate"] <= max_date)
].copy()

display(Markdown(f"* **Observation-window rows:** `{len(obs):,}`"))
display(Markdown(f"* **Prediction-window rows:** `{len(pred_window):,}`"))


**Results.** Cutoff at 2011-09-10: 641,995 observation-window rows, 161,006 prediction-window rows.

# 2. Feature Engineering (observation window only)

Every feature below is computed strictly from `obs` (data up to the cutoff). None may reference `pred_window` — that would be leakage.

In [ ]:
features = obs.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (CUTOFF_DATE - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("LineRevenue", "sum"),
    AvgBasketValue=("LineRevenue", lambda x: x.sum() / obs.loc[x.index, "Invoice"].nunique()),
    Tenure=("InvoiceDate", lambda x: (CUTOFF_DATE - x.min()).days),
    NDistinctProducts=("StockCode", "nunique"),
).reset_index()

# Cancellation rate as an auxiliary signal (notebook 02 established this is informative)
cancel_rate = clean[clean["InvoiceDate"] <= CUTOFF_DATE].groupby("Customer ID").agg(
    TotalInv=("Invoice", "nunique"),
    CancelledInv=("Invoice", lambda x: x[clean.loc[x.index, "IsCancellation"]].nunique()),
)
cancel_rate["CancellationRate"] = (cancel_rate["CancelledInv"] / cancel_rate["TotalInv"]).fillna(0)
features = features.merge(cancel_rate[["CancellationRate"]], on="Customer ID", how="left")
features["CancellationRate"] = features["CancellationRate"].fillna(0)

display(Markdown(f"**Customers with at least one observation-window purchase:** `{len(features):,}`"))
display(features.describe().round(2))


**Results.** 5,257 customers. No anomalies in the summary: `Frequency` and `Monetary` are strictly positive (unlike the full-dataset RFM table in notebook 02, every customer here has at least one observation-window purchase by construction), `Tenure` and `Recency` fall within sensible bounds (0-648 days), and `NDistinctProducts` reaches up to 2,180 for the largest wholesale accounts — consistent with the heavy-tailed customer base already seen throughout this project.

# 3. Target Construction

The target is binary: did the customer make at least one purchase in the 90-day prediction window?

In [ ]:
repurchasers = set(pred_window["Customer ID"].unique())
features["WillRepurchase"] = features["Customer ID"].isin(repurchasers).astype(int)

target_rate = features["WillRepurchase"].mean() * 100
display(Markdown(f"**Repurchase rate in the {PREDICTION_WINDOW_DAYS}-day window:** `{target_rate:.1f}%`"))
display(features["WillRepurchase"].value_counts())


**Results.** 43.6% repurchase rate (2,290 / 5,257) — close enough to balanced that ROC-AUC and PR-AUC both remain interpretable without heavy class-imbalance caveats.

# 4. Train / Test Split (time-based)

A random split would leak information: two purchases from the same customer close in time can end up on opposite sides of a random split, letting the model implicitly learn from a customer's own future behaviour. Instead, we split by an *earlier* cutoff within the observation window itself.

In [ ]:
TRAIN_CUTOFF_DAYS_BACK = 90
TRAIN_CUTOFF_DATE = CUTOFF_DATE - pd.Timedelta(TRAIN_CUTOFF_DAYS_BACK, unit="D")

obs_train_window = obs[obs["InvoiceDate"] <= TRAIN_CUTOFF_DATE]
pred_train_window = obs[
    (obs["InvoiceDate"] > TRAIN_CUTOFF_DATE) & (obs["InvoiceDate"] <= CUTOFF_DATE)
]

train_features = obs_train_window.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (TRAIN_CUTOFF_DATE - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("LineRevenue", "sum"),
    AvgBasketValue=("LineRevenue", lambda x: x.sum() / obs_train_window.loc[x.index, "Invoice"].nunique()),
    Tenure=("InvoiceDate", lambda x: (TRAIN_CUTOFF_DATE - x.min()).days),
    NDistinctProducts=("StockCode", "nunique"),
).reset_index()

train_repurchasers = set(pred_train_window["Customer ID"].unique())
train_features["WillRepurchase"] = train_features["Customer ID"].isin(train_repurchasers).astype(int)

FEATURE_COLS = ["Recency", "Frequency", "Monetary", "AvgBasketValue", "Tenure", "NDistinctProducts"]

X_train = train_features[FEATURE_COLS]
y_train = train_features["WillRepurchase"]

X_test = features[FEATURE_COLS]
y_test = features["WillRepurchase"]

display(Markdown(f"""
* **Training cutoff:** `{TRAIN_CUTOFF_DATE}` | **Training customers:** `{len(X_train):,}` | **Positive rate:** `{y_train.mean()*100:.1f}%`
* **Test cutoff:** `{CUTOFF_DATE}` | **Test customers:** `{len(X_test):,}` | **Positive rate:** `{y_test.mean()*100:.1f}%`
"""))


**Note on the train/test gap.** There is an 11.4-point difference in repurchase rate: 32.2% in training vs. 43.6% in test. This is consistent with seasonality — the test prediction window (Sep.-Dec. 2011) covers the pre-Christmas period for a giftware retailer, while the training window (June-Sep.) does not. The test-set evaluation below therefore reflects performance during a high-demand period specifically, and should not be assumed to generalize unchanged to a low-demand season without re-validation.

# 5. Baseline Models

In [ ]:
dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(X_train, y_train)

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

models = {"Dummy (stratified)": dummy, "Logistic Regression": logreg, "Random Forest": rf}


# 6. Model Comparison & Evaluation

ROC-AUC alone can be misleading on an imbalanced target — it is shown alongside PR-AUC, which is more sensitive to performance on the minority class.

In [ ]:
results = []
for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    results.append({"Model": name, "ROC-AUC": roc_auc, "PR-AUC": pr_auc})

results_df = pd.DataFrame(results)
display(Markdown("### Model comparison"))
display(results_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    axes[0].plot(fpr, tpr, label=name)
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    axes[1].plot(rec, prec, label=name)

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_title("ROC curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].axhline(y_test.mean(), color="k", linestyle="--", alpha=0.3, label="Baseline prevalence")
axes[1].set_title("Precision-Recall curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()
plt.tight_layout()
plt.show()


**Results.**

| Model | ROC-AUC | PR-AUC |
|---|---|---|
| Dummy (stratified) | 0.4967 | 0.4340 |
| Logistic Regression | **0.7903** | **0.7551** |
| Random Forest | 0.7751 | 0.7522 |

The dummy baseline scores almost exactly at chance (ROC-AUC ≈ 0.50, PR-AUC ≈ the 43.6% base rate), as expected — both models add real, substantial signal over it.

**Logistic Regression outperforms Random Forest** on both metrics, which is reported as-is rather than treated as a shortfall: with a moderate sample size (5,257 customers) and a plausibly monotonic relationship between features like Recency and repurchase likelihood, a simpler model can generalize better across a time-based split than an ensemble that has more room to overfit noise. **Logistic Regression is used as the reference model** for the remaining sections.

In [ ]:
best_model_name = results_df.sort_values("PR-AUC", ascending=False).iloc[0]["Model"]
best_model = models[best_model_name]

y_proba_best = best_model.predict_proba(X_test)[:, 1]
prob_true, prob_pred = calibration_curve(y_test, y_proba_best, n_bins=10)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(prob_pred, prob_true, marker="o", label=best_model_name)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfectly calibrated")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title(f"Calibration curve — {best_model_name}")
ax.legend()
plt.show()


**Action:** report whether the calibration curve sits close to the diagonal for Logistic Regression. Logistic Regression is typically well-calibrated by construction (it directly optimizes log-loss), so this is a useful sanity check rather than an open question — a curve that deviates noticeably from the diagonal would be a genuine surprise worth investigating.

# 7. Threshold Optimization

Instead of the default 0.5 cutoff, the decision threshold is chosen from an explicit business cost assumption.

In [ ]:
COST_OF_CONTACT = 2.0     # placeholder — replace with a real campaign cost
VALUE_OF_RETENTION = 25.0  # placeholder — replace with a real average order value

thresholds = np.linspace(0.05, 0.95, 19)
net_values = []

for t in thresholds:
    y_pred = (y_proba_best >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    net_value = tp * VALUE_OF_RETENTION - (tp + fp) * COST_OF_CONTACT
    net_values.append(net_value)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, net_values, marker="o")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Net value (£)")
ax.set_title("Net value by decision threshold")
plt.show()

best_threshold = thresholds[np.argmax(net_values)]
display(Markdown(f"**Threshold maximizing net value:** `{best_threshold:.2f}`"))


**Results.** The optimal threshold comes out at **0.05** — close to "contact almost everyone." This is a mechanical consequence of the placeholder cost assumptions, not a genuine business recommendation: with `COST_OF_CONTACT = £2` and `VALUE_OF_RETENTION = £25`, the break-even predicted probability is only 2/25 = 8%, and given a 43.6% overall repurchase rate, most customers clear this bar easily. **This result should not be read as "always contact everyone is optimal"** — it is an artifact of these specific, illustrative cost figures. Before using this threshold operationally, `COST_OF_CONTACT` and `VALUE_OF_RETENTION` must be replaced with real or at least defensible figures, and the threshold re-derived.

# 8. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind="barh", ax=ax)
ax.set_title("Random Forest feature importance")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

display(importances.round(4))


**Results.**

```
Recency              0.2479
Monetary             0.2042
AvgBasketValue       0.1433
NDistinctProducts    0.1428
Tenure               0.1403
Frequency            0.1215
```

`Recency` dominates, unsurprisingly — how long ago a customer last purchased is the single strongest signal for whether they'll purchase again soon. More interesting: **`Monetary` ranks second, ahead of `Frequency`** — how much a customer spends in total is a stronger repurchase signal than how often they order. For a business audience, this is the more useful takeaway: prioritizing outreach by spend level, not just order count, is better supported by this model than the more obvious "recency matters most" headline.